# YOLOv12 Training - Batu-Adit Pathole Dataset
## Google Colab Notebook | 5-Fold Cross Validation

| Hyperparameter | Nilai |
|---|---|
| Optimizer | **AdamW** |
| Learning Rate | 0.001 |
| Batch Size | 32 |
| Epoch | 200 |
| Image Size | 640 |
| K-Fold | 5 |
| Patience | 50 |

**Models:** YOLOv12 Default (n/s/m/l/x) + YOLOv12 GhostNet Backbone (n/s/m/l/x)

**Dataset:** Batu-Adit Pathole Dataset (nc=3: Besar, Kecil, Sedang)

**Metrik:** Precision, Recall, F1-Score, mAP50, mAP50-95, IoU, FLOPs, FPS, Train Time

> **Cara pakai:** Jalankan sel **Setup** (1-5) sekali di awal. Setelah itu,
> setiap sel model bisa dijalankan **satu per satu** secara independen.
> Ubah `START_FOLD` untuk melanjutkan training dari fold tertentu.

> **⚠️ PENTING:** Pastikan runtime Colab sudah diset ke **GPU** (Runtime → Change runtime type → GPU).
> Upload file `ultralytics_full.zip` dan `Batu-Adit.v1-pathole_dataset.yolov12.zip` ke Google Drive Anda.

---
# SETUP (Jalankan Sekali)


## 1. Cek Environment & GPU

In [ ]:
import torch, sys
print('Python  :', sys.executable)
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
torch.cuda.empty_cache()


## 1.5 Mount Google Drive
> Mount Google Drive untuk mengakses file zip dan menyimpan hasil training secara persisten.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ====================================
# KONFIGURASI PATH GOOGLE DRIVE
# ====================================
# Sesuaikan path ini dengan lokasi file di Google Drive Anda
GDRIVE_BASE = '/content/drive/MyDrive/SKRIPSI'  # <-- Ubah sesuai folder Anda di GDrive

# Path file zip di Google Drive
GDRIVE_ULTRALYTICS_ZIP = f'{GDRIVE_BASE}/ultralytics_full.zip'
GDRIVE_DATASET_ZIP     = f'{GDRIVE_BASE}/Batu-Adit.v1-pathole_dataset.yolov12.zip'

# Path untuk backup hasil training di Google Drive
GDRIVE_RESULTS = f'{GDRIVE_BASE}/results_colab'

import os
print(f'Google Drive mounted')
print(f'Ultralytics zip : {os.path.exists(GDRIVE_ULTRALYTICS_ZIP)}')
print(f'Dataset zip     : {os.path.exists(GDRIVE_DATASET_ZIP)}')


## 2. Install Ultralytics Custom
> Copy **ultralytics_full.zip** dari Google Drive ke `/content/` lalu install.

In [ ]:
import os, subprocess, zipfile, shutil

# Pastikan GDRIVE_ULTRALYTICS_ZIP sudah didefinisikan sebelumnya, 
# contoh: GDRIVE_ULTRALYTICS_ZIP = '/content/drive/MyDrive/ultralytics_full.zip'
ULTRALYTICS_ZIP = '/content/ultralytics_full.zip'
ULTRALYTICS_DIR = '/content/ultralytics'

# 1. Copy dari Google Drive jika belum ada di /content/
if not os.path.exists(ULTRALYTICS_ZIP) and os.path.exists(GDRIVE_ULTRALYTICS_ZIP):
    print('Copying ultralytics zip dari Google Drive...')
    shutil.copy(GDRIVE_ULTRALYTICS_ZIP, ULTRALYTICS_ZIP)
    print('Selesai copy')

# 2. Extract ZIP dan fix masalah backslash (\) dari Windows ke Linux (/)
if not os.path.exists(ULTRALYTICS_DIR):
    if os.path.exists(ULTRALYTICS_ZIP):
        print('Unzipping ultralytics...')
        with zipfile.ZipFile(ULTRALYTICS_ZIP, 'r') as z:
            for member in z.infolist():
                # Ubah pemisah folder agar valid di Linux (Colab)
                member.filename = member.filename.replace('\\', '/')
                z.extract(member, '/content')
        
        print('Selesai unzip')
        print("Isi /content/ultralytics:", os.listdir("/content/ultralytics")[:10] if os.path.exists("/content/ultralytics") else "Tidak ada")
    else:
        raise FileNotFoundError('ultralytics_full.zip tidak ditemukan di /content atau Google Drive')
else:
    print('Folder ultralytics sudah ada')

# 3. Install dependency (kita hapus '-e ULTRALYTICS_DIR' karena folder Anda hanya berisi package-nya saja, bukan root repo)
result = subprocess.run(['pip', 'install', 'scikit-learn'], capture_output=True, text=True)
print(result.stdout[-500:] if result.stdout else '')

# 4. Import & Verifikasi
import ultralytics
print('ultralytics loaded from:', ultralytics.__file__)

from ultralytics import YOLO
print('YOLO berhasil diimport dari ultralytics lokal')

# 5. Cek eksistensi GhostNet YAML dengan path yang benar
yaml_path = "/content/ultralytics/cfg/models/12/yolo12n-GhostNet.yaml"
print(f"GhostNet YAML exist? ", os.path.exists(yaml_path))


## 3. Setup Dataset
> Copy **Batu-Adit.v1-pathole_dataset.yolov12.zip** dari Google Drive ke `/content/`

In [ ]:
import os, zipfile, shutil

DATASET_ZIP = '/content/Batu-Adit.v1-pathole_dataset.yolov12.zip'
DATASET_DIR = '/content/dataset'

# Copy dari Google Drive jika belum ada di /content/
if not os.path.exists(DATASET_ZIP) and os.path.exists(GDRIVE_DATASET_ZIP):
    print('Copying dataset zip dari Google Drive...')
    shutil.copy(GDRIVE_DATASET_ZIP, DATASET_ZIP)
    print('Selesai copy')

if not os.path.exists(os.path.join(DATASET_DIR, 'train', 'images')):
    if os.path.exists(DATASET_ZIP):
        print('Unzipping dataset...')
        os.makedirs(DATASET_DIR, exist_ok=True)
        with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
            z.extractall(DATASET_DIR)
        print('Selesai unzip')
    else:
        print(f'Zip tidak ditemukan. Pastikan dataset ada di {DATASET_DIR}')
else:
    print('Dataset sudah tersedia')

IMAGE_DIR = os.path.join(DATASET_DIR, 'train', 'images')
if os.path.exists(IMAGE_DIR):
    imgs = [f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    print(f'Train images : {len(imgs)}')
    TEST_DIR = os.path.join(DATASET_DIR, 'test', 'images')
    if os.path.exists(TEST_DIR):
        ti = [f for f in os.listdir(TEST_DIR) if f.lower().endswith(('.jpg','.jpeg','.png'))]
        print(f'Test images  : {len(ti)}')
    print('Classes      : Besar, Kecil, Sedang (nc=3)')


## 4. Konversi Label: Polygon → YOLO Detection (BBox)

> Dataset dilabeli menggunakan **Smart Select (Polygon/Segmentation)**.
> Sel ini mengkonversi format polygon ke format **YOLO Detection (bounding box)** 
> sebelum training. **Wajib dijalankan sekali** setelah dataset di-unzip.
>
> ⚠️ Jangan jalankan sel ini dua kali — label akan tertimpa (idempotent tidak dijamin).

In [ ]:
import os

ALL_LABEL_DIRS = [
    '/content/dataset/train/labels',
    '/content/dataset/test/labels',
]

total_converted = 0
total_skipped   = 0

for LABEL_DIR in ALL_LABEL_DIRS:
    if not os.path.exists(LABEL_DIR):
        print(f'Folder tidak ditemukan, dilewati: {LABEL_DIR}')
        continue
    print(f'\nProcessing: {LABEL_DIR}')
    dir_converted = 0
    for file in os.listdir(LABEL_DIR):
        if not file.endswith('.txt'): continue
        path = os.path.join(LABEL_DIR, file)
        new_lines = []
        with open(path, 'r') as f:
            lines = f.readlines()
        for line in lines:
            parts = line.strip().split()
            if len(parts) == 5:
                new_lines.append(line)
                continue
            if len(parts) < 7:
                total_skipped += 1
                continue
            cls = parts[0]
            coords = list(map(float, parts[1:]))
            xs = coords[0::2]
            ys = coords[1::2]
            x_center = (min(xs) + max(xs)) / 2
            y_center = (min(ys) + max(ys)) / 2
            width    = max(xs) - min(xs)
            height   = max(ys) - min(ys)
            new_line = f'{cls} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n'
            new_lines.append(new_line)
            dir_converted += 1
        with open(path, 'w') as f:
            f.writelines(new_lines)
    print(f'  Converted : {dir_converted} polygon annotations')
    total_converted += dir_converted

print(f'\nTotal converted : {total_converted}')
print(f'Total skipped   : {total_skipped}')


## 4.5 Cek Normalisasi Bounding Box (0..1)

In [ ]:
import glob, random
paths = glob.glob("/content/dataset/train/labels/*.txt")
if not paths:
    print('Label files not found')
else:
    for p in random.sample(paths, min(5, len(paths))):
        with open(p) as f:
            parts = f.readline().strip().split()
        if len(parts) == 5:
            vals = list(map(float, parts[1:]))
            print(vals, "OK" if all(0 <= v <= 1 for v in vals) else "NOT OK")


## 5. Verifikasi Label Setelah Konversi

In [ ]:
import os

LABEL_DIR = '/content/dataset/train/labels'
corrupt_files = []
total_files = 0

for file in os.listdir(LABEL_DIR):
    if not file.endswith('.txt'): continue
    total_files += 1
    with open(os.path.join(LABEL_DIR, file), 'r') as f:
        lines = f.readlines()
    for line in lines:
        parts = line.strip().split()
        if len(parts) != 5:
            corrupt_files.append((file, f'Kolom: {len(parts)}'))
            continue
        try:
            cls, x, y, w, h = map(float, parts)
            if int(cls) not in range(3):
                corrupt_files.append((file, f'Class invalid: {int(cls)}'))
        except Exception as ex:
            corrupt_files.append((file, str(ex)))

print(f'Total label files : {total_files}')
print(f'Corrupt           : {len(corrupt_files)}')
if not corrupt_files:
    print('Semua label valid!')
for item in corrupt_files[:20]:
    print(' Warning:', item)


---
# TRAINING MODEL

Setiap sel di bawah ini **independen** dan bisa dijalankan satu per satu.
Ubah `START_FOLD = 1` untuk mulai dari awal, atau angka lain untuk resume.


---
## 🔷 YOLOv12 Default Backbone


### YOLOv12n (nano)

In [ ]:
import ultralytics
print('ultralytics loaded from:', ultralytics.__file__)
from ultralytics import YOLO
import yaml
import os
import time
import shutil
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold

try:
    def smart_link(src, dst):
        try:
            if os.path.lexists(dst):
                os.remove(dst)
            os.symlink(src, dst)
        except Exception:
            shutil.copy(src, dst)

    # =============================
    # 1. KONFIGURASI
    # =============================
    print("\n" + "="*70)
    print("K-FOLD TRAINING : YOLOv12n (nano)")
    print("="*70)

    IMAGE_DIR   = "/content/dataset/train/images"
    LABEL_DIR   = "/content/dataset/train/labels"
    BASE_OUTPUT = "/content/results"

    MODEL       = "/content/ultralytics/cfg/models/12/yolo12n.yaml"
    FOLDER_NAME = "Yolo12n"

    EPOCHS        = 200
    BATCH_SIZE    = 32
    IMAGE_SIZE    = 640
    LEARNING_RATE = 0.001
    OPTIMIZER     = "Adam"
    NUM_FOLDS     = 5
    START_FOLD    = 1   # <-- Ganti angka ini untuk resume dari fold tertentu
    PATIENCE      = 50

    CLASS_NAMES = ['Besar', 'Kecil', 'Sedang']
    NC          = len(CLASS_NAMES)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device : {device}")

    # =============================
    # 2. AMBIL DATA GAMBAR
    # =============================
    image_files = sorted([
        f for f in os.listdir(IMAGE_DIR)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])
    print(f"Total images : {len(image_files)}")

    # =============================
    # 3. HITUNG FLOPs & PARAMS
    # =============================
    print("\nMenghitung FLOPs & parameter model...")
    _tmp_model = YOLO(MODEL)

    model_flops = 0.0
    try:
        _info = _tmp_model.info(verbose=False, imgsz=IMAGE_SIZE)
        if isinstance(_info, (list, tuple)) and len(_info) >= 4:
            model_flops = round(float(_info[3]), 2)
    except Exception:
        pass

    total_params     = sum(p.numel() for p in _tmp_model.model.parameters())
    trainable_params = sum(p.numel() for p in _tmp_model.model.parameters() if p.requires_grad)
    del _tmp_model
    torch.cuda.empty_cache()

    print(f"FLOPs            : {model_flops} GFLOPs")
    print(f"Total params     : {total_params:,}")
    print(f"Trainable params : {trainable_params:,}")

    # =============================
    # 4. K-FOLD SETUP
    # =============================
    kf = KFold(
        n_splits=NUM_FOLDS,
        shuffle=True,
        random_state=42
    )

    all_results = []

    # =============================
    # 5. LOOP CROSS VALIDATION
    # =============================
    for fold, (train_idx, val_idx) in enumerate(kf.split(image_files)):

        if fold + 1 < START_FOLD:
            continue

        print("\n" + "="*70)
        print(f"\U0001f680 TRAINING FOLD {fold+1}/{NUM_FOLDS}")
        print("="*70)

        fold_dir      = f"/tmp/fold_{fold+1}"
        train_img_dir = os.path.join(fold_dir, "train/images")
        train_lbl_dir = os.path.join(fold_dir, "train/labels")
        val_img_dir   = os.path.join(fold_dir, "valid/images")
        val_lbl_dir   = os.path.join(fold_dir, "valid/labels")

        os.makedirs(train_img_dir, exist_ok=True)
        os.makedirs(train_lbl_dir, exist_ok=True)
        os.makedirs(val_img_dir,   exist_ok=True)
        os.makedirs(val_lbl_dir,   exist_ok=True)

        # COPY TRAIN FILES
        for idx in train_idx:
            img_name   = image_files[idx]
            label_name = Path(img_name).stem + ".txt"
            smart_link(os.path.join(IMAGE_DIR, img_name),
                        os.path.join(train_img_dir, img_name))
            src_lbl = os.path.join(LABEL_DIR, label_name)
            if os.path.exists(src_lbl):
                smart_link(src_lbl, os.path.join(train_lbl_dir, label_name))

        # COPY VALID FILES
        for idx in val_idx:
            img_name   = image_files[idx]
            label_name = Path(img_name).stem + ".txt"
            smart_link(os.path.join(IMAGE_DIR, img_name),
                        os.path.join(val_img_dir, img_name))
            src_lbl = os.path.join(LABEL_DIR, label_name)
            if os.path.exists(src_lbl):
                smart_link(src_lbl, os.path.join(val_lbl_dir, label_name))

        print(f"Train: {len(train_idx)} images | Val: {len(val_idx)} images")

        # CREATE data.yaml
        data_yaml = {
            'path' : fold_dir,
            'train': 'train/images',
            'val'  : 'valid/images',
            'nc'   : NC,
            'names': CLASS_NAMES
        }
        yaml_path = os.path.join(fold_dir, "data.yaml")
        with open(yaml_path, 'w') as f:
            yaml.safe_dump(data_yaml, f)

        # LOAD / RESUME MODEL
        model_name  = Path(MODEL).stem
        output_dir  = os.path.join(BASE_OUTPUT, FOLDER_NAME)
        resume_path = os.path.join(
            output_dir, f"{model_name}_fold_{fold+1}",
            "weights", "last.pt"
        )

        if os.path.exists(resume_path):
            print(f"\n\U0001f504 Resume training Fold {fold+1}")
            model  = YOLO(resume_path)
            RESUME = True
        else:
            print(f"\n\U0001f195 Training baru Fold {fold+1}")
            model  = YOLO(MODEL)
            RESUME = False

        print("\n\U0001f9e0 MODEL INFO")
        print("-"*70)
        print(f"Total params      : {total_params:,}")
        print(f"Trainable params  : {trainable_params:,}")
        print(f"FLOPs             : {model_flops} GFLOPs")

        # TRAINING
        start_time = time.time()

        results_train = model.train(
            data      = yaml_path,
            epochs    = EPOCHS,
            batch     = BATCH_SIZE,
            imgsz     = IMAGE_SIZE,
            optimizer = OPTIMIZER,
            lr0       = LEARNING_RATE,
            project   = output_dir,
            name      = f"{model_name}_fold_{fold+1}",
            exist_ok  = True,
            device    = 0 if torch.cuda.is_available() else 'cpu',
            verbose   = True,
            patience  = PATIENCE,
            save      = True,
            resume    = RESUME,
            amp       = True
        )

        train_time_sec = time.time() - start_time
        train_time_min = train_time_sec / 60.0

        run_dir         = Path(results_train.save_dir)
        best_model_path = run_dir / "weights/best.pt"

        print(f"\n\u2705 Fold {fold+1} selesai")
        print(f"Training time : {train_time_min:.2f} menit ({train_time_sec/3600:.2f} jam)")

        # VALIDASI
        model_best  = YOLO(best_model_path)
        val_results = model_best.val(
            data  = yaml_path,
            split = "val",
            plots = False
        )

        # FPS
        print("\n\U0001f3a5 MENGHITUNG FPS")

        val_images = [
            os.path.join(val_img_dir, f)
            for f in os.listdir(val_img_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]

        _ = model_best.predict(val_images[0], verbose=False)

        inference_times = []
        for img_path in val_images:
            start_inf = time.time()
            _ = model_best.predict(
                source = img_path,
                imgsz  = IMAGE_SIZE,
                verbose= False,
                device = 0 if torch.cuda.is_available() else 'cpu'
            )
            inference_times.append(time.time() - start_inf)

        avg_latency = np.mean(inference_times)
        fps         = 1 / avg_latency

        # IoU (Mean IoU prediksi vs GT)
        print("\n\U0001f4d0 MENGHITUNG MEAN IoU")

        from ultralytics.utils.metrics import box_iou

        iou_list   = []
        _val_imgs  = val_images[:100]

        for _img_path in _val_imgs:
            _lbl_path = os.path.join(
                val_lbl_dir, Path(_img_path).stem + ".txt"
            )
            if not os.path.exists(_lbl_path):
                continue

            _res = model_best.predict(
                source=_img_path, imgsz=IMAGE_SIZE,
                verbose=False,
                device=0 if torch.cuda.is_available() else 'cpu'
            )[0]

            if _res.boxes is None or len(_res.boxes) == 0:
                continue

            _h, _w = _res.orig_shape
            _gt    = []

            with open(_lbl_path) as _f:
                for _line in _f:
                    _p = _line.strip().split()
                    if len(_p) != 5: continue
                    _, _xc, _yc, _bw, _bh = map(float, _p)
                    _gt.append([
                        (_xc - _bw/2) * _w, (_yc - _bh/2) * _h,
                        (_xc + _bw/2) * _w, (_yc + _bh/2) * _h
                    ])

            if not _gt:
                continue

            import torch as _torch
            _pred_xy = _res.boxes.xyxy.cpu()
            _gt_t    = _torch.tensor(_gt, dtype=_torch.float32)
            _iou_mat = box_iou(_pred_xy, _gt_t)
            if _iou_mat.numel() > 0:
                iou_list.extend(_iou_mat.max(dim=1)[0].tolist())

        mean_iou = round(float(np.mean(iou_list)), 4) if iou_list else 0.0

        # METRIK
        P     = val_results.results_dict.get('metrics/precision(B)', 0)
        R     = val_results.results_dict.get('metrics/recall(B)', 0)
        m50   = val_results.results_dict.get('metrics/mAP50(B)', 0)
        m5095 = val_results.results_dict.get('metrics/mAP50-95(B)', 0)
        F1    = 2 * P * R / (P + R) if (P + R) > 0 else 0

        print("\n\U0001f4ca HASIL FOLD")
        print("-"*70)
        print(f"Precision    : {P:.4f}")
        print(f"Recall       : {R:.4f}")
        print(f"F1-Score     : {F1:.4f}")
        print(f"mAP50        : {m50:.4f}")
        print(f"mAP50-95     : {m5095:.4f}")
        print(f"Mean IoU     : {mean_iou:.4f}")
        print(f"FLOPs        : {model_flops} GFLOPs")
        print(f"FPS          : {fps:.2f}")
        print(f"Train Time   : {train_time_min:.2f} menit")

        # SAVE CSV
        csv_path = run_dir / "results.csv"
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            df["total_params"]     = total_params
            df["trainable_params"] = trainable_params
            df["flops_gflops"]     = model_flops
            df["fps"]              = fps
            df["mean_iou"]         = mean_iou
            df["train_time_min"]   = train_time_min
            df.to_csv(csv_path, index=False)
            print("\u2713 results.csv updated")

        # SIMPAN HASIL FOLD
        all_results.append([
            fold + 1, P, R, F1, m50, m5095,
            mean_iou, model_flops, fps,
            total_params, train_time_min
        ])

        # Bersihkan folder tmp fold
        shutil.rmtree(fold_dir, ignore_errors=True)
        torch.cuda.empty_cache()

        # BACKUP KE GOOGLE DRIVE (setiap fold)
        try:
            import shutil as _shutil
            _gdrive_dst = f"{GDRIVE_RESULTS}/{FOLDER_NAME}"
            os.makedirs(_gdrive_dst, exist_ok=True)
            _src_fold = os.path.join(output_dir, f"{model_name}_fold_{fold+1}")
            if os.path.exists(_src_fold):
                _dst_fold = os.path.join(_gdrive_dst, f"{model_name}_fold_{fold+1}")
                if os.path.exists(_dst_fold):
                    _shutil.rmtree(_dst_fold)
                _shutil.copytree(_src_fold, _dst_fold)
                print(f"\u2713 Fold {fold+1} backed up to Google Drive")
        except Exception as _bk_err:
            print(f"\u26a0 Backup error: {_bk_err}")

    # =============================
    # 6. HASIL AKHIR (SUMMARY)
    # =============================
    print("\n" + "="*70)
    print("\U0001f4c8 CROSS VALIDATION SUMMARY")
    print("="*70)

    result_df = pd.DataFrame(
        all_results,
        columns=[
            "Fold", "Precision", "Recall", "F1-Score",
            "mAP50", "mAP50-95", "Mean_IoU",
            "FLOPs_GFLOPs", "FPS",
            "Total_Params", "Train_Time_min"
        ]
    )

    print(result_df.to_string(index=False))

    print("\n\U0001f4ca RATA-RATA HASIL")
    print("-"*70)
    print(result_df.mean(numeric_only=True).to_string())

    # Simpan summary CSV
    os.makedirs(os.path.join(BASE_OUTPUT, FOLDER_NAME), exist_ok=True)
    summary_csv = os.path.join(BASE_OUTPUT, FOLDER_NAME, 'summary_Yolo12n.csv')
    result_df.to_csv(summary_csv, index=False)
    print(f"\n\u2713 Summary saved : {summary_csv}")

    # Backup summary ke Google Drive
    try:
        import shutil as _shutil
        _gdrive_dst = f"{GDRIVE_RESULTS}/{FOLDER_NAME}"
        os.makedirs(_gdrive_dst, exist_ok=True)
        _shutil.copy(summary_csv, os.path.join(_gdrive_dst, 'summary_Yolo12n.csv'))
        print("\u2713 Summary backed up to Google Drive")
    except Exception as _bk_err:
        print(f"\u26a0 Backup error: {_bk_err}")

    # =============================
    # 7. TABEL HASIL CROSS VALIDATION (FORMAT SKRIPSI)
    #    Nilai desimal x100 = persentase, koma desimal, Mean +/- SD
    # =============================
    print("\n" + "="*70)
    print("\U0001f4cb TABEL HASIL CROSS VALIDATION (Format Skripsi)")
    print("="*70)

    def _fmt_pct(val, decimals=4):
        """Format angka ke string dengan koma desimal (Indonesia)"""
        return f'{val:.{decimals}f}'.replace('.', ',')

    def _fmt_mean_sd(values, decimals=2):
        """Hitung Mean +/- SD dan format dengan koma desimal"""
        m = np.mean(values)
        s = np.std(values, ddof=1)  # sample std deviation
        m_str = f'{m:.{decimals}f}'.replace('.', ',')
        s_str = f'{s:.{decimals}f}'.replace('.', ',')
        return f'{m_str} \u00b1 {s_str}'

    pct_cols = ['Precision', 'Recall', 'F1-Score', 'mAP50', 'mAP50-95']

    header = f"{'Fold':<6} {'Precision (%)':<16} {'Recall (%)':<14} {'F1-Score (%)':<15} {'mAP@0.5 (%)':<15} {'mAP@0.5:0.95 (%)':<19} {'FPS':<10}"
    print(header)
    print("-" * len(header))

    pct_data = {col: [] for col in pct_cols}
    fps_data = []

    for _, row in result_df.iterrows():
        fold_num = int(row['Fold'])
        p_pct   = row['Precision'] * 100
        r_pct   = row['Recall'] * 100
        f1_pct  = row['F1-Score'] * 100
        m50_pct = row['mAP50'] * 100
        m95_pct = row['mAP50-95'] * 100
        fps_val = row['FPS']

        pct_data['Precision'].append(p_pct)
        pct_data['Recall'].append(r_pct)
        pct_data['F1-Score'].append(f1_pct)
        pct_data['mAP50'].append(m50_pct)
        pct_data['mAP50-95'].append(m95_pct)
        fps_data.append(fps_val)

        print(f'{fold_num:<6} {_fmt_pct(p_pct):<16} {_fmt_pct(r_pct):<14} {_fmt_pct(f1_pct):<15} {_fmt_pct(m50_pct):<15} {_fmt_pct(m95_pct):<19} {_fmt_pct(fps_val):<10}')

    print("-" * len(header))
    mean_sd_row = (
        f"{'Mean':<6} "
        f"{_fmt_mean_sd(pct_data['Precision']):<16} "
        f"{_fmt_mean_sd(pct_data['Recall']):<14} "
        f"{_fmt_mean_sd(pct_data['F1-Score']):<15} "
        f"{_fmt_mean_sd(pct_data['mAP50']):<15} "
        f"{_fmt_mean_sd(pct_data['mAP50-95']):<19} "
        f"{_fmt_mean_sd(fps_data):<10}"
    )
    print(mean_sd_row)

    # Simpan tabel skripsi sebagai CSV
    skripsi_rows = []
    for _, row in result_df.iterrows():
        skripsi_rows.append({
            'Fold': int(row['Fold']),
            'Precision (%)': round(row['Precision'] * 100, 4),
            'Recall (%)': round(row['Recall'] * 100, 4),
            'F1-Score (%)': round(row['F1-Score'] * 100, 4),
            'mAP@0.5 (%)': round(row['mAP50'] * 100, 4),
            'mAP@0.5:0.95 (%)': round(row['mAP50-95'] * 100, 4),
            'FPS': round(row['FPS'], 4),
        })

    skripsi_df = pd.DataFrame(skripsi_rows)
    mean_vals = {}
    for col in ['Precision (%)', 'Recall (%)', 'F1-Score (%)', 'mAP@0.5 (%)', 'mAP@0.5:0.95 (%)', 'FPS']:
        m = round(skripsi_df[col].mean(), 2)
        s = round(skripsi_df[col].std(ddof=1), 2)
        mean_vals[col] = f'{m} \u00b1 {s}'

    mean_row = {'Fold': 'Mean \u00b1 SD'}
    mean_row.update(mean_vals)
    skripsi_full = pd.concat([skripsi_df, pd.DataFrame([mean_row])], ignore_index=True)

    skripsi_csv = os.path.join(BASE_OUTPUT, FOLDER_NAME, 'tabel_skripsi_Yolo12n.csv')
    skripsi_full.to_csv(skripsi_csv, index=False)
    print(f"\n\u2713 Tabel skripsi saved : {skripsi_csv}")

    # Backup tabel skripsi ke Google Drive
    try:
        import shutil as _shutil2
        _gdrive_dst2 = f"{GDRIVE_RESULTS}/{FOLDER_NAME}"
        os.makedirs(_gdrive_dst2, exist_ok=True)
        _shutil2.copy(skripsi_csv, os.path.join(_gdrive_dst2, 'tabel_skripsi_Yolo12n.csv'))
        print("\u2713 Tabel skripsi backed up to Google Drive")
    except Exception as _bk_err2:
        print(f"\u26a0 Backup error: {_bk_err2}")

except Exception as e:
    print(f"\n\u274c Error : {e}")
    import traceback
    traceback.print_exc()
